In [ ]:
import sys
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("ReadSingleCSV") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

In [ ]:
CSV_DF = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(r"C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv")
).where(col("CabNumber") == 'T802127C')


In [22]:
CSV_DF.show(5)

+---------+--------------------+----------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+-------+--------------------+---------------+
|CabNumber|VehicleLicenseNumber|      Name|     LicenseType|Active|PermitLicenseNumber| VehicleVinNumber|WheelchairAccessible|VehicleYear|VehicleType|TelephoneNumber|Website|             Address|LastDateUpdated|
+---------+--------------------+----------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+-------+--------------------+---------------+
| T802127C|              C19641|ABCON INC.|OWNER MUST DRIVE|   YES|               NULL|5TDBK3EH0DS268018|                NULL|       2016|       NULL|  (718)438-1100|   NULL|41-24   38 STREET...|     04/22/2020|
+---------+--------------------+----------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+--

In [ ]:
LicenseType_DF = CSV_DF.withColumn("LicenseType", split("LicenseType", "")).selectExpr("LicenseType")

In [25]:
LicenseType_DF.show(5)

+--------------------+
|         LicenseType|
+--------------------+
|[O, W, N, E, R,  ...|
+--------------------+



In [26]:
LicenseType_DFExplode = LicenseType_DF.select(explode("LicenseType").alias("LicenseType"))

In [30]:
LicenseType_DFExplode.show()

+-----------+
|LicenseType|
+-----------+
|          O|
|          W|
|          N|
|          E|
|          R|
|           |
|          M|
|          U|
|          S|
|          T|
|           |
|          D|
|          R|
|          I|
|          V|
|          E|
+-----------+



In [ ]:
LicenseType_DFExplodex = LicenseType_DFExplode.where(col("LicenseType") != "").groupBy("LicenseType").count()
LicenseType_DFExplodex.show()

SparkSteam >>>>>

In [43]:
Vehicle_DF = spark.readStream.format("socket").option("host", "localhost").option("port", 9999).load()

In [44]:
Vehicle_DFx = Vehicle_DF.withColumn("LicenseType", split(col("value"), "")).select("LicenseType")

In [45]:
Vehicle_DFExplode = Vehicle_DFx.select(explode("LicenseType").alias("LicenseType"))

In [46]:
Vehicle_DFGrouped = Vehicle_DFExplode.where(col("LicenseType") != "").groupBy("LicenseType").count()

In [47]:
Vehicle_DFGrouped.writeStream.outputMode("complete").format("console").start()